# YOLOv8s-P2 300 Epoch Training on Kaggle

基于原始 10k 数据集 notebook 改写，使用 `YOLOv8s + P2` 进行长训。


In [ ]:
# Cell 1: 克隆代码（直接使用 GitHub 最新版本）
%cd /kaggle/working
!rm -rf yolov8s_kuangjing
!git clone https://github.com/tuanziya666/yolov8s_kuangjing.git
%cd /kaggle/working/yolov8s_kuangjing
!git pull


In [ ]:
# Cell 2: 基础环境检查，并关闭 wandb/raytune
%cd /kaggle/working/yolov8s_kuangjing
!python -c "import torch, cv2, numpy; from ultralytics import YOLO; print('OK'); print('torch=', torch.__version__); print('cv2=', cv2.__version__); print('numpy=', numpy.__version__)"
!python -c "from ultralytics.utils import SETTINGS; SETTINGS.update(wandb=False, raytune=False); print(SETTINGS)"


In [ ]:
# Cell 3: 查看 Kaggle 数据集挂载路径
!find /kaggle/input -maxdepth 3 -type d | head -n 100


In [ ]:
# Cell 4: 设置新数据集路径并写出 Kaggle 专用 data.yaml
# 把下面这个路径改成你上传到 Kaggle 的 10k 数据集根目录
DATASET_ROOT = "/kaggle/input/datasets/yuanfangshang/yolo-kuangjing-hard15k-dataset"

from pathlib import Path

yaml_text = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner
"""

cfg_path = Path('/kaggle/working/yolov8s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml')
cfg_path.write_text(yaml_text, encoding='utf-8')
print(cfg_path.read_text(encoding='utf-8'))


In [ ]:
# Cell 5: 确认 P2 模型配置和数据配置文件
%cd /kaggle/working/yolov8s_kuangjing
!ls -lh ultralytics/cfg/models/v8/yolov8s_p2.yaml
!cat ultralytics/cfg/datasets/coalmine4_kaggle.yaml


In [ ]:
!find /kaggle/input/datasets/yuanfangshang/yolo-kuangjing-mixed25k-dataset -maxdepth 2 -type f | head -n 100
!find /kaggle/input/datasets/yuanfangshang/yolo-kuangjing-mixed25k-dataset -maxdepth 2 -type d | head -n 100


In [ ]:
# 新增 Cell：让 DDP 子进程也能找到本地 ultralytics
%cd /kaggle/working/yolov8s_kuangjing

import os
import sys

REPO_ROOT = "/kaggle/working/yolov8s_kuangjing"
os.environ["PYTHONPATH"] = REPO_ROOT + ":" + os.environ.get("PYTHONPATH", "")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# 关键：只做可编辑安装，不装依赖，避免再把 numpy 环境装坏
!python -m pip install -q --no-deps -e /kaggle/working/yolov8s_kuangjing

# 检查一下当前导入的就是你本地这份代码
!python -c "import ultralytics, sys; print('python=', sys.executable); print('ultralytics=', ultralytics.__file__)"


In [ ]:
# Cell 6: 训练 YOLOv8s-P2（300 epoch，固定随机种子）
%cd /kaggle/working/yolov8s_kuangjing

import random
import numpy as np
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RUN_NAME = 'yolov8s_p2_10k_300ep'
RUN_PROJECT = '/kaggle/working/runs'

model = YOLO('ultralytics/cfg/models/v8/yolov8s_p2.yaml')
model.train(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    task='detect',
    project=RUN_PROJECT,
    name=RUN_NAME,
    device='0,1',
    epochs=300,
    batch=72,
    imgsz=640,
    workers=6,
    seed=SEED,
    deterministic=True,
    pretrained='yolov8s.pt',
    save=True,
    val=True,
    plots=True,
    verbose=True,
    exist_ok=False,
)


In [ ]:
# Cell 7: 用 best.pt 分别在 val / test 上重新评估并保存结果
%cd /kaggle/working/yolov8s_kuangjing

from ultralytics import YOLO

RUN_NAME = 'yolov8s_p2_10k_300ep'
BEST = f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt'

model = YOLO(BEST)
print('BEST =', BEST)

print('===== VALIDATE ON VAL =====')
model.val(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='val',
    imgsz=640,
    batch=64,
    device='0,1',
    plots=True,
    project='/kaggle/working/evals',
    name=f'{RUN_NAME}_val',
)

print('===== VALIDATE ON TEST =====')
model.val(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='test',
    imgsz=640,
    batch=32,
    device='0',
    plots=True,
    project='/kaggle/working/evals',
    name=f'{RUN_NAME}_test',
)


In [ ]:
# Cell 8: 整理一个误差库（test 集预测可视化 + GT 标签 + 预测标签）
%cd /kaggle/working/yolov8s_kuangjing

from pathlib import Path
import shutil
from ultralytics import YOLO

RUN_NAME = 'yolov8s_p2_10k_300ep'
BEST = Path(f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt')
ERROR_ROOT = Path('/kaggle/working/error_bank')
GT_LABELS = Path(DATASET_ROOT) / 'labels' / 'test'
TEST_IMAGES = Path(DATASET_ROOT) / 'images' / 'test'

if ERROR_ROOT.exists():
    shutil.rmtree(ERROR_ROOT)
ERROR_ROOT.mkdir(parents=True, exist_ok=True)

model = YOLO(str(BEST))
model.predict(
    source=str(TEST_IMAGES),
    imgsz=640,
    conf=0.25,
    device=0,
    project=str(ERROR_ROOT),
    name='test_predictions',
    save=True,
    save_txt=True,
    save_conf=True,
)

GT_DST = ERROR_ROOT / 'test_gt_labels'
GT_DST.mkdir(parents=True, exist_ok=True)
for p in GT_LABELS.glob('*.txt'):
    shutil.copy2(p, GT_DST / p.name)

README = ERROR_ROOT / 'README.txt'
README.write_text(
    'error_bank contents:
'
    '- test_predictions/: 模型在 test 集上的可视化预测和 pred labels
'
    '- test_gt_labels/: test 集对应的 GT labels
'
    '- 结合 evals/ 下的 confusion_matrix 和 PR 曲线一起看误差
',
    encoding='utf-8'
)
print('Error bank saved to', ERROR_ROOT)


In [ ]:
# Cell 9: 查看核心输出文件
RUN_NAME = 'yolov8s_p2_10k_300ep'
!ls /kaggle/working/runs/{RUN_NAME}
!ls /kaggle/working/runs/{RUN_NAME}/weights
!ls /kaggle/working/evals/{RUN_NAME}_val
!ls /kaggle/working/evals/{RUN_NAME}_test
!ls /kaggle/working/error_bank


In [ ]:
# Cell 10: 打包训练结果、评估结果和误差库，方便下载
RUN_NAME = 'yolov8s_p2_10k_300ep'
!zip -r /kaggle/working/{RUN_NAME}.zip /kaggle/working/runs/{RUN_NAME}
!zip -r /kaggle/working/{RUN_NAME}_evals.zip /kaggle/working/evals/{RUN_NAME}_val /kaggle/working/evals/{RUN_NAME}_test
!zip -r /kaggle/working/{RUN_NAME}_error_bank.zip /kaggle/working/error_bank
